# AAR Legibility PVG: experiments (v5)

v4 (Brandon's main + `apply_fixes.py`) with three changes:

1. Warm start for 3 epochs (v4 used 2 and reached helpful fidelity 0.27-0.41, under the 0.5 target).
2. Five seeds (42-46).
3. Sneaky prover lines on the plot (sneaky success = flawed AND accepted, sneaky accepted, sneaky fidelity), recomputed from the saved samples, with round 7 marked and a table of the rise (round 1 to 7) and the decline (round 7 to 10).

Results are copied to Google Drive (`MyDrive/pvg_v5_results`) as well as downloaded, so `Analysis_v5_claude_compare.ipynb` can read them from there.

Runtime: A100. About 15 minutes of setup, about 10 minutes of warm start per seed, then 6 to 9 minutes per round: roughly 7 to 8 hours for 5 seeds x 10 rounds. Every cell after the pilot is resumable.

## 1. Setup

In [ ]:
import os
from pathlib import Path

REPO_URL  = "https://github.com/gunasti2002/Automated_R_leg_NoAPI.git"
BRANCH    = "main"
CLONE_DIR = Path("/content/Automated_R_leg_NoAPI")

%cd /content
!rm -rf {CLONE_DIR}
!git clone --branch {BRANCH} {REPO_URL} {CLONE_DIR}

matches = sorted(CLONE_DIR.rglob("config.py"))
assert matches, f"No config.py found under {CLONE_DIR}"
PROJECT_ROOT = matches[0].parent
os.chdir(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)
!git -C {CLONE_DIR} log --oneline -3


In [ ]:
%%writefile pvg_extra.py
"""
Per-round metrics recomputed from the saved prover samples, including the sneaky
prover lines that round_history.jsonl does not log, and a plot that marks round 7.

    from pvg_extra import load_samples, round_metrics, plot_rounds, rise_and_decline
    df = round_metrics(load_samples("training/checkpoint"))

A write-up counts as ACCEPTED when the verifier's P(sound) at reward time is >= 0.5,
and as FLAWED when it fails the rule check (the project's ground truth).

  helpful_accept     helpful write-ups accepted (= helpful_prover_accuracy in the logs)
  helpful_fidelity   helpful write-ups that pass the rule check
  sneaky_accept      sneaky write-ups accepted (= 1 - verifier_robustness)
  sneaky_success     sneaky write-ups that are FLAWED and still accepted, over all sneaky
                     write-ups (Kirchner et al.'s sneaky success rate)
  sneaky_fool_rate   accepted, among the FLAWED sneaky write-ups only
  sneaky_fidelity    sneaky write-ups that are flawed
"""
import json
from pathlib import Path

import numpy as np
import pandas as pd

ACCEPT = 0.5


def _f(x):
    return float(x) if not isinstance(x, str) else float(x)


def _b(x):
    return x if isinstance(x, bool) else str(x).lower() == "true"


def load_samples(root) -> pd.DataFrame:
    rows = []
    for f in sorted(Path(root).glob("seed_*/samples/round_*.jsonl")):
        for line in f.read_text().splitlines():
            if not line.strip():
                continue
            s = json.loads(line)
            rows.append({"seed": int(s["seed"]), "round": int(s["round_number"]), "role": s["label"],
                         "p_sound": _f(s["verifier_p_sound"]), "rule_ok": _b(s["rule_check_passes"]),
                         "summary": s["summary"], "experiment": s["experiment"]})
    if not rows:
        raise SystemExit(f"No sample files under {root}/seed_*/samples/. Point this at the run's checkpoint folder "
                         f"or the unzipped pvg_results folder.")
    return pd.DataFrame(rows)


def round_metrics(samples: pd.DataFrame) -> pd.DataFrame:
    out = []
    for (seed, rnd), g in samples.groupby(["seed", "round"]):
        h, s = g[g.role == "helpful"], g[g.role == "sneaky"]
        acc_h, acc_s = h.p_sound >= ACCEPT, s.p_sound >= ACCEPT
        flawed = ~s.rule_ok
        out.append({"seed": seed, "round": rnd,
                    "helpful_accept": acc_h.mean(), "helpful_fidelity": h.rule_ok.mean(),
                    "sneaky_accept": acc_s.mean(), "sneaky_success": (acc_s & flawed).mean(),
                    "sneaky_fool_rate": acc_s[flawed].mean() if flawed.any() else np.nan,
                    "sneaky_fidelity": flawed.mean(),
                    "reward_gap": h.p_sound.mean() - s.p_sound.mean(),
                    "n_helpful": len(h), "n_sneaky": len(s)})
    return pd.DataFrame(out)


LINES = [("helpful_accept", "helpful accepted (helpful accuracy)", "tab:blue"),
         ("sneaky_success", "sneaky success (flawed AND accepted)", "tab:red"),
         ("sneaky_accept", "sneaky accepted (1 - robustness)", "tab:orange"),
         ("helpful_fidelity", "helpful passes rule check", "tab:green"),
         ("sneaky_fidelity", "sneaky is flawed", "tab:purple"),
         ("reward_gap", "reward gap (mean P(sound) helpful - sneaky)", "tab:gray")]


def plot_rounds(df: pd.DataFrame, out=None, mark_round=7, title="PVG rounds"):
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True)
    groups = [LINES[:3], LINES[3:]]
    n_seeds = df.seed.nunique()
    for ax, group, sub in zip(axes, groups, ("Verifier decisions", "Prover behaviour")):
        for key, label, color in group:
            g = df.groupby("round")[key]
            mean, lo, hi, n = g.mean(), g.min(), g.max(), g.count()
            ax.plot(mean.index, mean.values, marker="o", color=color, label=label)
            ax.fill_between(mean.index, lo.values, hi.values, color=color, alpha=0.15)
        if mark_round:
            ax.axvline(mark_round, ls="--", color="k", lw=1)
            ax.text(mark_round + 0.1, 1.02, f"round {mark_round}", fontsize=9)
        ax.set_ylim(-0.02, 1.08); ax.set_xlabel("PVG round"); ax.set_title(sub); ax.grid(alpha=0.3)
        ax.legend(fontsize=8, loc="lower right")
    axes[0].set_ylabel(f"rate (mean over {n_seeds} seeds; band = min/max)")
    fig.suptitle(title)
    fig.tight_layout()
    if out:
        fig.savefig(out, dpi=150)
    return fig


def per_round_table(df: pd.DataFrame) -> pd.DataFrame:
    keys = [k for k, _, _ in LINES] + ["sneaky_fool_rate"]
    t = df.groupby("round")[keys].agg(["mean", "std"])
    t.columns = [f"{a}_{b}" for a, b in t.columns]
    t["seeds"] = df.groupby("round").seed.nunique()
    return t.round(3)


def rise_and_decline(df: pd.DataFrame, peak=7) -> pd.DataFrame:
    """Round 1 -> round `peak` (the rise) and round `peak` -> last round (the decline), mean over seeds."""
    last = int(df["round"].max())
    keys = [k for k, _, _ in LINES]
    m = df.groupby("round")[keys].mean()
    rows = []
    for k in keys:
        r1 = m.loc[1, k] if 1 in m.index else np.nan
        rp = m.loc[peak, k] if peak in m.index else np.nan
        rl = m.loc[last, k]
        rows.append({"metric": k, "round 1": r1, f"round {peak}": rp, f"round {last}": rl,
                     f"rise 1->{peak}": rp - r1, f"change {peak}->{last}": rl - rp})
    return pd.DataFrame(rows).set_index("metric").round(3)


In [ ]:
# Colab ships torchao 0.10, which peft refuses (needs >= 0.16). Nothing here uses it: remove it.
!pip uninstall -y -q torchao
# torch + matplotlib ship with Colab; do not reinstall torch (CUDA wheel mismatch risk).
!pip install -q jinja2 "transformers>=4.50" peft accelerate
import importlib; importlib.invalidate_caches()

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv


### Brandon's fixes

Pins the project code to 51940d1, patches it, rebuilds the dataset with the 10 real records and writes the Qwen-written rows (GPU, about 15 minutes). In the output, check that every patch line starts with `+` and that both model-written files have rows. The data cell below rebuilds the same dataset, which is harmless.

In [ ]:
!python apply_fixes.py
!git -C {CLONE_DIR} status --short | head -20

In [ ]:
# Brandon's change 3: a stronger learning signal for the prover. Subprocesses inherit this.
import os, json
OVERRIDES = {
    "prover_lr": 1e-4,                 # was 2e-5
    "prover_num_generations": 8,       # was 4 samples per record
    "findings_per_round": 16,          # was 8 records per round
    "prover_kl_penalty": 0.05,         # was 0.02
    # Both roles gated by the rule check: helpful earns P(sound) only if it passes,
    # sneaky only if it fails. Matches the verifier labels (sneaky write-ups that pass
    # the rule check are dropped from verifier training, so they should earn nothing).
    "prover_reward_mode": "correctness_gated",
    # v5: 3 warm-start epochs (apply_fixes default is 2; v4 reached helpful fidelity 0.27-0.41)
    "prover_sft_epochs": 3,
}
os.environ["PVG_CONFIG_OVERRIDES"] = json.dumps(OVERRIDES)

import sys; sys.path.insert(0, ".")
from config import PVGConfig
cfg = PVGConfig()
print({k: getattr(cfg, k) for k in OVERRIDES})
print("samples per round:", 2 * cfg.findings_per_round * cfg.prover_num_generations)


## 2. Data

120 honest/sneaky pairs, 20% held out by record, seed 42 (unchanged).

In [ ]:
!python data/build_findings_dataset.py --target-pairs 120 --seed 42 \
    --out data/findings_pvg_dataset.jsonl --heldout-out data/findings_pvg_heldout.jsonl \
    --manifest-out data/dataset_manifest.json | tail -4


## 3. Does the untrained prover write findings that pass the rule check?

Two minutes, no training. This is the check for change 1. If helpful fidelity is below 0.5
here, the helpful prover will earn almost no reward and the rest of the run repeats the
26 Sep failure, so fix the prompt before going further. The script prints why the failing
write-ups fail and shows two of them.

This is base Qwen, before the warm start. The pilot log prints the warm-started number as `[sft] after warm start: helpful fidelity`, and that one should be at least 0.5.

In [ ]:
!python eval/prover_fidelity.py --records 8 --generations 4 --min-helpful 0.5


In [ ]:
# Verifier spot check (untrained) and the two baselines on the held-out split.
from training import pvg_loop
spot_rows = pvg_loop.load_dataset(cfg.spot_check_set_path)
report = pvg_loop.run_spot_check(cfg, spot_set_rows=spot_rows, enforce_set=False)
!python eval/baselines.py --out eval/baselines_heldout.json | grep -E "^\["


In [ ]:
# Drop the notebook's verifier so the training subprocesses get the full card.
import gc, sys, torch
if "training.pvg_loop" in sys.modules:
    sys.modules["training.pvg_loop"]._VERIFIER_CACHE.clear()
gc.collect(); torch.cuda.empty_cache()
free, total = torch.cuda.mem_get_info()
print(f"GPU free: {free/1024**3:.2f} / {total/1024**3:.2f} GiB")
assert free / total > 0.9, "Something is still holding GPU memory — restart the runtime."


## 4. Pilot: seed 42, 3 rounds, then check the four targets

The pilot writes to the same checkpoint folder as the full run, so the full run in
section 5 continues seed 42 from round 4 instead of repeating these rounds.
Exit 2 = the spot-check gate refused to train. Exit 3 = the seed aborted itself.

In [ ]:
SEEDS          = [42, 43, 44, 45, 46]
TOTAL_ROUNDS   = 10
PILOT_ROUNDS   = 3
CHECKPOINT_DIR = "training/checkpoint"

PEAK_ROUND     = 7      # marked on the plots; the rise is round 1 -> 7, the decline 7 -> last
DRIVE_RESULTS  = "/content/drive/MyDrive/pvg_v5_results"


In [ ]:
!python run_experiment.py --seeds 42 --rounds {PILOT_ROUNDS} --checkpoint-dir {CHECKPOINT_DIR} --no-plot


In [ ]:
# Brandon's four targets, checked per seed from round_history.jsonl.
import json
from pathlib import Path

def load_history(ckpt):
    out = {}
    for f in sorted(Path(ckpt).glob("seed_*/round_history.jsonl")):
        rows = [json.loads(l) for l in f.read_text().splitlines() if l.strip()]
        if rows: out[rows[0].get("seed", f.parent.name)] = rows
    return out

def check_targets(ckpt):
    hist = load_history(ckpt)
    if not hist:
        print("No completed rounds yet."); return
    for seed, rows in hist.items():
        r1, last = rows[0], rows[-1]
        fid = [r["role_fidelity_helpful"] for r in rows]
        gap = [r["reward_gap"] for r in rows]
        bias = [r.get("verifier_logit_bias") or 0.0 for r in rows]
        acc = [r["helpful_prover_accuracy"] for r in rows]
        share = [r.get("verifier_train_sound_share") for r in rows]
        checks = [
            ("helpful fidelity >= 0.5 in round 1 and not falling", fid[0] >= 0.5 and fid[-1] >= fid[0] - 0.1,
             "  ".join(f"{x:.2f}" for x in fid)),
            ("reward gap stays positive", min(gap) > 0, "  ".join(f"{x:+.2f}" for x in gap)),
            ("logit bias within +/-3", max(abs(b) for b in bias) <= 3.0, "  ".join(f"{x:+.1f}" for x in bias)),
            ("helpful accuracy does not crash after round 1", min(acc[1:] or acc) >= 0.5,
             "  ".join(f"{x:.2f}" for x in acc)),
        ]
        print(f"seed {seed}: {len(rows)} round(s)" + ("  ABORTED: " + last.get("notes", "") if last.get("aborted") else ""))
        for name, ok, vals in checks:
            print(f"  [{'PASS' if ok else 'FAIL'}] {name:52s} {vals}")
        print("         SOUND share of verifier training pool      ", "  ".join("n/a" if s is None else f"{s:.2f}" for s in share))
        print("         held-out accuracy / AUROC                  ",
              "  ".join(f"{r['heldout_accuracy']:.2f}/{r['heldout_auroc']:.2f}" for r in rows))

check_targets(CHECKPOINT_DIR)


In [ ]:
# What the helpful prover actually wrote in the latest round: failures with the rule check's reasons.
import json
from pathlib import Path
files = sorted(Path(CHECKPOINT_DIR).glob("seed_42/samples/round_*.jsonl"))
if files:
    rows = [json.loads(l) for l in files[-1].read_text().splitlines() if l.strip()]
    h = [r for r in rows if r["label"] == "helpful"]
    fails = [r for r in h if not r["rule_check_passes"]]
    print(f"{files[-1].name}: {len(h)} helpful samples, {len(fails)} fail the rule check\n")
    for r in fails[:4]:
        print(json.dumps(r["experiment"]["metrics"]))
        print("  issues:", r["consistency_issues"])
        print("  " + r["summary"][:600].replace("\n", " ") + "\n")
else:
    print("No sample files yet.")


## 5. Full run and results

Continues seed 42 from where the pilot stopped, then runs seeds 43 to 46. Safe to interrupt; re-run the cell to resume. The results are copied to Drive after the run (and can be copied any time with the save cell).

In [ ]:
seeds = " ".join(str(s) for s in SEEDS)
!python run_experiment.py --seeds {seeds} --rounds {TOTAL_ROUNDS} --checkpoint-dir {CHECKPOINT_DIR}


In [ ]:
import subprocess, sys
from pathlib import Path
png = Path(CHECKPOINT_DIR) / "pvg_curves.png"
subprocess.run([sys.executable, "eval/plot_rounds.py", "--checkpoint-dir", CHECKPOINT_DIR, "--out", str(png)])
if png.exists():
    from IPython.display import Image, display
    display(Image(filename=str(png)))
check_targets(CHECKPOINT_DIR)


### Sneaky prover lines, round 7 marked

Recomputed from `samples/round_NN.jsonl`. Accepted = the verifier's P(sound) at reward time is at least 0.5; flawed = fails the rule check. `helpful_accept` matches `helpful_prover_accuracy` in the logs and `sneaky_accept` matches `1 - verifier_robustness`.

In [ ]:
from pvg_extra import load_samples, round_metrics, plot_rounds, per_round_table, rise_and_decline
from IPython.display import display
samples = load_samples(CHECKPOINT_DIR)
metrics = round_metrics(samples)
metrics.to_csv(f"{CHECKPOINT_DIR}/round_metrics_from_samples.csv", index=False)
fig = plot_rounds(metrics, out=f"{CHECKPOINT_DIR}/pvg_curves_with_sneaky.png", mark_round=PEAK_ROUND,
                  title=f"v5: {metrics.seed.nunique()} seeds")
display(fig)
print("Rise (round 1 -> 7) and decline (round 7 -> last), mean over seeds:")
display(rise_and_decline(metrics, peak=PEAK_ROUND))
print("Per round, mean and std over seeds:")
display(per_round_table(metrics))

In [ ]:
import json, pandas as pd
from pathlib import Path
rows = []
for f in sorted(Path(CHECKPOINT_DIR).glob("seed_*/round_history.jsonl")):
    rows += [json.loads(l) for l in f.read_text().splitlines() if l.strip()]
df = pd.DataFrame(rows)
if df.empty:
    print("No completed rounds yet.")
cols = ["seed", "round_number", "helpful_prover_accuracy", "verifier_robustness", "role_fidelity_helpful",
        "role_fidelity_sneaky", "reward_gap", "helpful_mean_reward", "sneaky_mean_reward",
        "heldout_accuracy", "heldout_auroc", "heldout_accept_rate", "verifier_logit_bias",
        "verifier_train_sound_share", "verifier_relabeled_helpful", "verifier_dropped_sneaky", "aborted", "elapsed_s"]
df[[c for c in cols if c in df.columns]]


### Save the results before the runtime ends

Copies the histories, samples, plots, tables and warm-start reports (not the model weights) to `MyDrive/pvg_v5_results` and downloads a zip. The v4 zip never reached the laptop, so the Drive copy is the one to rely on.

In [ ]:
import shutil, os
from pathlib import Path
out = Path("/content/pvg_results"); shutil.rmtree(out, ignore_errors=True); out.mkdir()
for f in Path(CHECKPOINT_DIR).rglob("*"):
    if f.is_file() and f.suffix in (".jsonl", ".json", ".png", ".csv") and not set(f.parts) & {
            "verifier", "helpful_prover", "sneaky_prover", "prover_sft_base"} and f.name != "replay_buffer.json":
        dest = out / f.relative_to(CHECKPOINT_DIR); dest.parent.mkdir(parents=True, exist_ok=True); shutil.copy(f, dest)
for f in ("eval/baselines_heldout.json", "eval/prover_fidelity.json", "data/dataset_manifest.json"):
    if Path(f).exists(): shutil.copy(f, out / Path(f).name)
for f in Path(CHECKPOINT_DIR).glob("seed_*/prover_sft_base/sft_report.json"):
    dest = out / f.relative_to(CHECKPOINT_DIR); dest.parent.mkdir(parents=True, exist_ok=True); shutil.copy(f, dest)
shutil.make_archive("/content/pvg_results", "zip", out)
print("files:", sum(1 for _ in out.rglob("*") if _.is_file()), "| archive MB:", round(os.path.getsize("/content/pvg_results.zip") / 1e6, 2))

# 1. Google Drive (survives the runtime; the analysis notebook reads it from here)
from google.colab import drive
drive.mount("/content/drive")
shutil.rmtree(DRIVE_RESULTS, ignore_errors=True)
shutil.copytree(out, DRIVE_RESULTS)
shutil.copy("/content/pvg_results.zip", DRIVE_RESULTS + ".zip")
print("copied to", DRIVE_RESULTS, "and", DRIVE_RESULTS + ".zip")

# 2. Download to this computer as well
from google.colab import files
files.download("/content/pvg_results.zip")